In [ ]:
import os

from sketch2sim.agents.descriptor_agent import build_descriptor_graph, DescriptorState
from sketch2sim.agents.description_validator_agent import (
    build_description_validator_graph,
    DescriptionValidatorState,
)
from sketch2sim.agents.extractor_agent import build_extractor_graph, ExtractorState
from sketch2sim.mixing_points_normalizer import normalize_mixing_points
from sketch2sim.coupling_normalizer import couple_extraction

from sketch2sim.agents.basis_agent import build_basis_graph, BasisState
from sketch2sim.agents.instantiation_agent import build_instantiation_graph, InstantiationState
from sketch2sim.agents.configuration_agent import build_configuration_graph, ConfigurationState

from sketch2sim.agents.fixer_agent import fix_code_file
from sketch2sim.agents.executor_agent import build_executor_graph, ExecutorState
from sketch2sim.merge_simulation_scripts import merge_basis_and_configuration

from sketch2sim.utils import (
    ensure_dir,
    log_text,
    log_json,
    log_agent_step,
    is_success,
    finish_success,
    run_step,
    run_stage,
    merge_execute_and_log,
    handle_runtime_failure_and_fix,
    build_unit_view_from_structure,
    run_merge_execute_with_runtime_fixer,
)

"""
Main orchestrator for the diagram-to-simulation workflow.
"""

if not os.getenv("OLLAMA_API_KEY"):
    raise EnvironmentError("OLLAMA_API_KEY is not set.")

LOG_DIR = "logs"
OUTPUT_DIR = "outputs"

IMAGE_PATH = r"images/cs1.png"

DESCRIPTOR_MODEL = "gemini-3-flash-preview:cloud"
DESCRIPTION_VALIDATOR_MODEL = "qwen3-vl:8b-instruct-q4_K_M"
EXTRACTOR_MODEL = "gemini-3-flash-preview:cloud"

BASIS_MODEL = "qwen2.5-coder:latest"
INSTANTIATION_MODEL = "qwen2.5-coder:latest"
CONFIGURATION_MODEL = "qwen3-coder:30b-a3b-q4_K_M"

BASIS_TEMPLATE_PATH = "sketch2sim/instructions/hysys_template_base.py"
INSTANTIATION_INSTRUCTION_FILES = ["sketch2sim/instructions/instantiation_instructions_simple_v5.txt"]
CONFIGURATION_INSTRUCTION_FILES = ["sketch2sim/instructions/configuration_instructions_simple_v5.txt"]

MAX_RUNTIME_FIX_RETRIES = 3

STEP = {
    "descriptor": "[1️⃣ Descriptor Agent] Running...",
    "description_validator": "[2️⃣ Description Validator] Running...",
    "extractor": "[3️⃣ Extractor] Running...",
    "basis": "[5️⃣ Basis Agent] Generating Basis Script...",
    "instantiation": "[6️⃣ Instantiation Agent] Generating Unit Operations + Streams...",
    "normalization": "[7️⃣ Normalization] Building Unit IN/OUT View...",
    "configuration": "[8️⃣ Configuration Agent] Connecting Streams...",
}


def main():
    ensure_dir(LOG_DIR)
    ensure_dir(OUTPUT_DIR)

    # STEP 1 Descriptor
    descriptor_output = run_step(
        STEP["descriptor"],
        build_descriptor_graph(DESCRIPTOR_MODEL),
        DescriptorState(image_path=IMAGE_PATH),
    )
    descriptor_description = (descriptor_output.get("description") or "").strip()
    log_text(LOG_DIR, "step1_descriptor_result.txt", descriptor_description)

    # STEP 2 Description Validator
    description_validation_output = run_step(
        STEP["description_validator"],
        build_description_validator_graph(DESCRIPTION_VALIDATOR_MODEL),
        DescriptionValidatorState(description=descriptor_description),
    )
    description_validation = description_validation_output.get("validation", {})
    log_json(LOG_DIR, "step2_description_validation.json", description_validation)

    # STEP 3 Extractor - processes descriptor description only
    extractor_output = run_step(
        STEP["extractor"],
        build_extractor_graph(EXTRACTOR_MODEL),
        ExtractorState(process_text=descriptor_description),
    )
    extraction = extractor_output.get("extraction", extractor_output)
    log_json(LOG_DIR, "step3_extraction.json", extraction or {})

    # STEP 4A Mixing Points Normalization
    normalized_structure = normalize_mixing_points(extraction)
    log_json(LOG_DIR, "step4A_mixing_points_normalized.json", normalized_structure or {})

    # STEP 4B Coupling Normalization
    coupled_structure = couple_extraction(normalized_structure)
    structure_json_str = log_json(
        LOG_DIR,
        "step4B_coupled_structure.json",
        coupled_structure or {},
    )

    # STEP 5 Basis
    basis_path = run_stage(
        label=STEP["basis"],
        step="step5_basis",
        graph=build_basis_graph(BASIS_MODEL),
        state=BasisState(
            description=descriptor_description,
            template_path=BASIS_TEMPLATE_PATH,
            structure_json=structure_json_str,
        ),
        prefix="hysys_basis",
        log_dir=LOG_DIR,
        output_dir=OUTPUT_DIR,
        log_agent_step_fn=log_agent_step,
    )

    # STEP 6 Instantiation
    instantiation_path = run_stage(
        label=STEP["instantiation"],
        step="step6_instantiation",
        graph=build_instantiation_graph(INSTANTIATION_MODEL),
        state=InstantiationState(
            description=structure_json_str,
            basis_path=basis_path,
            instruction_paths=INSTANTIATION_INSTRUCTION_FILES,
        ),
        prefix="hysys_instantiation",
        log_dir=LOG_DIR,
        output_dir=OUTPUT_DIR,
        log_agent_step_fn=log_agent_step,
    )

    # STEP 7 Unit View Normalization
    unit_view_json_str = log_json(
        LOG_DIR,
        "step7_unit_view.json",
        build_unit_view_from_structure(coupled_structure or {}),
    )

    # STEP 8 Configuration
    configuration_path = run_stage(
        label=STEP["configuration"],
        step="step8_configuration",
        graph=build_configuration_graph(CONFIGURATION_MODEL),
        state=ConfigurationState(
            description=unit_view_json_str,
            instantiation_path=instantiation_path,
            instruction_paths=CONFIGURATION_INSTRUCTION_FILES,
        ),
        prefix="hysys_configuration",
        log_dir=LOG_DIR,
        output_dir=OUTPUT_DIR,
        log_agent_step_fn=log_agent_step,
    )

    # STEP 9 Merge + Execute with runtime-driven fixer retries
    executor_graph = build_executor_graph()

    run_merge_execute_with_runtime_fixer(
        basis_path=basis_path,
        configuration_path=configuration_path,
        combined_path=os.path.join(OUTPUT_DIR, "hysys_combined_latest.py"),
        executor_graph=executor_graph,
        unit_view_json_str=unit_view_json_str,
        instantiation_instruction_paths=INSTANTIATION_INSTRUCTION_FILES,
        configuration_instruction_paths=CONFIGURATION_INSTRUCTION_FILES,
        fixer_model_name=CONFIGURATION_MODEL,
        log_dir=LOG_DIR,
        max_retries=MAX_RUNTIME_FIX_RETRIES,
        merge_fn=merge_basis_and_configuration,
        merge_execute_and_log=merge_execute_and_log,
        handle_runtime_failure_and_fix=handle_runtime_failure_and_fix,
        fixer_fn=fix_code_file,
        output_dir=OUTPUT_DIR,
        ExecutorState=ExecutorState,
        is_success=is_success,
        finish_success=finish_success,
    )


if __name__ == "__main__":
    main()